In [9]:
import pandas as pd

# Load the Excel file
file_path = 'medical_clean_d603.xlsx' 
data = pd.read_excel(file_path)

# Display the first few rows of the data
print(data.head())


   CaseOrder Customer_id                           Interaction  \
0          1     C412403  8cd49b13-f45a-4b47-a2bd-173ffa932c2f   
1          2     Z919181  d2450b70-0337-4406-bdbb-bc1037f1734c   
2          3     F995323  a2057123-abf5-4a2c-abad-8ffe33512562   
3          4     A879973  1dec528d-eb34-4079-adce-0d7a40e82205   
4          5     C544523  5885f56b-d6da-43a3-8760-83583af94266   

                                UID          City State        County    Zip  \
0  3a83ddb66e2ae73798bdf1d705dc0932           Eva    AL        Morgan  35621   
1  176354c5eef714957d486009feabf195      Marianna    FL       Jackson  32446   
2  e19a0fa00aeda885b8a436757e889bc9   Sioux Falls    SD     Minnehaha  57110   
3  cd17d7b6d152cb6f23957346d11c3f07  New Richland    MN        Waseca  56072   
4  d2f0425877b10ed6bb381f3e2579424a    West Point    VA  King William  23181   

        Lat       Lng  ...  TotalCharge Additional_charges Item1 Item2  Item3  \
0  34.34960 -86.72508  ...  3726.702860  

In [10]:
# Missing values 
data.fillna(method='ffill', inplace=True)

C:\Users\warre\AppData\Local\Temp\ipykernel_24244\1586891110.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data.fillna(method='ffill', inplace=True)


In [1]:
import pandas as pd

# Load the Excel file
file_path = 'medical_clean_d603.xlsx' 
data = pd.read_excel(file_path)

# Fill missing values
data.fillna(method='ffill', inplace=True)

# Identify binary and nominal categorical columns
binary_cols = ['Gender', 'HighBlood', 'Stroke', 'Complication_risk', 'Overweight', 'Arthritis', 'Diabetes']
nominal_cols = ['Initial_admin', 'Services']

# Label encode binary columns
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in binary_cols:
    data[col] = le.fit_transform(data[col])

# One-hot encode nominal columns
data = pd.get_dummies(data, columns=nominal_cols, drop_first=True)

# Select only the features you want (update as needed)
selected_features = [
    'Age', 'Initial_days', 'TotalCharge', 'Additional_charges', 'VitD_levels', 
    'Doc_visits', 'Gender', 'HighBlood', 'Stroke', 'Complication_risk', 
    'Overweight', 'Arthritis', 'Diabetes'
] + [col for col in data.columns if col.startswith('Initial_admin_') or col.startswith('Services_')]

X = data[selected_features]
y = data['ReAdmis']

# Save cleaned dataset
cleaned_data = pd.concat([X, y], axis=1)
cleaned_data.to_csv('cleaned_medical_data_d603task1.csv', index=False)

C:\Users\warre\AppData\Local\Temp\ipykernel_1844\1575404898.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data.fillna(method='ffill', inplace=True)


In [4]:
# Select all columns that start with 'Initial_admin_' or 'Services_' for one-hot encoded features
onehot_cols = [col for col in data.columns if col.startswith('Initial_admin_') or col.startswith('Services_')]

selected_features = [
    'Age', 'Initial_days', 'TotalCharge', 'Additional_charges', 'VitD_levels', 
    'Doc_visits', 'Gender', 'HighBlood', 'Stroke', 'Complication_risk', 
    'Overweight', 'Arthritis', 'Diabetes'
] + onehot_cols

X = data[selected_features]
y = data['ReAdmis']

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Save the datasets 
X_train.to_csv('d603task1_X_train.csv', index=False)
X_val.to_csv('d603task1_X_val.csv', index=False)
X_test.to_csv('d603task1_X_test.csv', index=False)
y_train.to_csv('d603task1_y_train.csv', index=False)
y_val.to_csv('d603task1_y_val.csv', index=False)
y_test.to_csv('d603task1_y_test.csv', index=False)

In [6]:
data.to_csv('cleaned_medical_data_d603task1.csv', index=False)

Training Random Forest Model

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Train the model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate on validation set
y_pred = rf_model.predict(X_val)
print(classification_report(y_val, y_pred))
print(confusion_matrix(y_val, y_pred))

              precision    recall  f1-score   support

          No       0.98      0.99      0.99       987
         Yes       0.98      0.97      0.97       513

    accuracy                           0.98      1500
   macro avg       0.98      0.98      0.98      1500
weighted avg       0.98      0.98      0.98      1500

[[975  12]
 [ 15 498]]


Performing k-fold cross validation to optimize hyperparameters

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],  # Number of trees in the forest
    'max_depth': [10, 20, None],     # Maximum depth of each tree
    'min_samples_split': [2, 5, 10]  # Minimum samples required to split a node
}

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='accuracy',  # Optimize for accuracy
    verbose=1,  # Display progress
    n_jobs=-1   # Use all available CPU cores
)

# Fit the model on the training data
grid_search.fit(X_train, y_train)

# Output the best parameters
print("Best Parameters:", grid_search.best_params_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best Parameters: {'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 300}


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Train the initial Random Forest model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Initial Model Evaluation on Validation Dataset
print("Initial Model (Validation Dataset):")
y_val_pred_initial = rf_model.predict(X_val)
y_val_pred_proba_initial = rf_model.predict_proba(X_val)[:, 1]
print(classification_report(y_val, y_val_pred_initial))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_val_pred_initial))
print("AUC-ROC:", roc_auc_score(y_val, y_val_pred_proba_initial))

# Optimized Model Evaluation on Test Dataset
print("\nOptimized Model (Test Dataset):")
optimized_model = grid_search.best_estimator_  # Ensure grid_search is already run
y_test_pred_optimized = optimized_model.predict(X_test)
y_test_pred_proba_optimized = optimized_model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_test_pred_optimized))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred_optimized))
print("AUC-ROC:", roc_auc_score(y_test, y_test_pred_proba_optimized))

Initial Model (Validation Dataset):
              precision    recall  f1-score   support

          No       0.98      0.99      0.98       987
         Yes       0.97      0.97      0.97       513

    accuracy                           0.98      1500
   macro avg       0.98      0.98      0.98      1500
weighted avg       0.98      0.98      0.98      1500

Confusion Matrix:
 [[973  14]
 [ 17 496]]
AUC-ROC: 0.9982452190365592

Optimized Model (Test Dataset):
              precision    recall  f1-score   support

          No       0.98      0.99      0.99       947
         Yes       0.99      0.97      0.98       553

    accuracy                           0.98      1500
   macro avg       0.98      0.98      0.98      1500
weighted avg       0.98      0.98      0.98      1500

Confusion Matrix:
 [[939   8]
 [ 16 537]]
AUC-ROC: 0.9989058433312774


In [18]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Load the saved training set
X_train = pd.read_csv('d603task1_X_train.csv')
y_train = pd.read_csv('d603task1_y_train.csv').squeeze()  # .squeeze() to convert DataFrame to Series

# Train the initial Random Forest model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Predict on training set
y_train_pred = rf_model.predict(X_train)
y_train_pred_proba = rf_model.predict_proba(X_train)[:, 1]

# Print metrics
print("Classification Report (Training Set):")
print(classification_report(y_train, y_train_pred))  # includes accuracy, precision, recall, F1
print("Confusion Matrix (Training Set):")
print(confusion_matrix(y_train, y_train_pred))
print("AUC-ROC (Training Set):", roc_auc_score(y_train, y_train_pred_proba))

Classification Report (Training Set):
              precision    recall  f1-score   support

          No       1.00      1.00      1.00      4397
         Yes       1.00      1.00      1.00      2603

    accuracy                           1.00      7000
   macro avg       1.00      1.00      1.00      7000
weighted avg       1.00      1.00      1.00      7000

Confusion Matrix (Training Set):
[[4397    0]
 [   0 2603]]
AUC-ROC (Training Set): 1.0


In [19]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Train the initial Random Forest model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Initial Model Evaluation on Training Dataset
print("Initial Model (Training Dataset):")
y_train_pred_initial = rf_model.predict(X_train)
y_train_pred_proba_initial = rf_model.predict_proba(X_train)[:, 1]
print(classification_report(y_train, y_train_pred_initial))
print("Confusion Matrix:\n", confusion_matrix(y_train, y_train_pred_initial))
print("AUC-ROC:", roc_auc_score(y_train, y_train_pred_proba_initial))

# Optimized Model Evaluation on Test Dataset
print("\nOptimized Model (Test Dataset):")
optimized_model = grid_search.best_estimator_  # Ensure grid_search is already run
y_test_pred_optimized = optimized_model.predict(X_test)
y_test_pred_proba_optimized = optimized_model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_test_pred_optimized))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred_optimized))
print("AUC-ROC:", roc_auc_score(y_test, y_test_pred_proba_optimized))

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Initial Model (Training Dataset):
              precision    recall  f1-score   support

          No       1.00      1.00      1.00      4397
         Yes       1.00      1.00      1.00      2603

    accuracy                           1.00      7000
   macro avg       1.00      1.00      1.00      7000
weighted avg       1.00      1.00      1.00      7000

Confusion Matrix:
 [[4397    0]
 [   0 2603]]
AUC-ROC: 1.0

Optimized Model (Test Dataset):
              precision    recall  f1-score   support

          No       0.98      0.99      0.99       947
         Yes       0.99      0.97      0.98       553

    accuracy                           0.98      1500
   macro avg       0.98      0.98      0.98      1500
weighted avg       0.98      0.98      0.98      1500

Confusion Matrix:
 [[939   8]
 [ 16 537]]
AUC-ROC: 0.9989058433312774
